In [0]:
# Load CSV files

orders=spark.read.csv("/Workspace/Users/s.dk2004th@gmail.com/SURE ProED/A10- Databricks One-Day Practical Assignment/orders.csv",header=True,inferSchema=True)
products=spark.read.csv("/Workspace/Users/s.dk2004th@gmail.com/SURE ProED/A10- Databricks One-Day Practical Assignment/products.csv",header=True,inferSchema=True)
aisles=spark.read.csv("/Workspace/Users/s.dk2004th@gmail.com/SURE ProED/A10- Databricks One-Day Practical Assignment/aisles.csv",header=True,inferSchema=True)
departments=spark.read.csv("/Workspace/Users/s.dk2004th@gmail.com/SURE ProED/A10- Databricks One-Day Practical Assignment/departments.csv",header=True,inferSchema=True)
order_train=spark.read.csv("/Workspace/Users/s.dk2004th@gmail.com/SURE ProED/A10- Databricks One-Day Practical Assignment/order_products__train.csv",header=True,inferSchema=True)
prior1=spark.read.csv("/Workspace/Users/s.dk2004th@gmail.com/SURE ProED/A10- Databricks One-Day Practical Assignment/order_products__prior - 1.csv",header=True,inferSchema=True)
prior2=spark.read.csv("/Workspace/Users/s.dk2004th@gmail.com/SURE ProED/A10- Databricks One-Day Practical Assignment/order_products__prior 2.csv",header=True,inferSchema=True)

# Combine both prior files
order_prior=prior1.union(prior2)

In [0]:
print("Orders")
orders.show(5)

print("Products")
products.show(5)

print("Prior Orders")
order_prior.show(5)

Orders
+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
| 2539329|      1|   prior|           1|        2|                8|                  NULL|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|
|  473747|      1|   prior|           3|        3|               12|                  21.0|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|
|  431534|      1|   prior|           5|        4|               15|                  28.0|
+--------+-------+--------+------------+---------+-----------------+----------------------+
only showing top 5 rows
Products
+----------+--------------------+--------+-------------+
|product_id|        product_name|aisle_id|department_id|
+----------+------

## Data Understanding

In [0]:
# Number of Rows
print("Orders:", orders.count())
print("Products:", products.count())
print("Aisles:", aisles.count())
print("Departments:", departments.count())
print("Order Train:", order_train.count())
print("Order Prior:", order_prior.count())

Orders: 3421083
Products: 49688
Aisles: 134
Departments: 21
Order Train: 1384617
Order Prior: 1048575


In [0]:
orders.printSchema()
products.printSchema()
aisles.printSchema()
departments.printSchema()
order_train.printSchema()
order_prior.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)

root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)

root
 |-- department_id: integer (nullable = true)
 |-- department: string (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- 

In [0]:
# Missing Values
from pyspark.sql.functions import col, sum

def check_nulls(df, name):
    print(f"\n{name}")
    df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).show()

check_nulls(orders, "Orders")
check_nulls(products, "Products")
check_nulls(aisles, "Aisles")
check_nulls(departments, "Departments")
check_nulls(order_train, "Order Train")
check_nulls(order_prior, "Order Prior")


Orders
+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                206209|
+--------+-------+--------+------------+---------+-----------------+----------------------+


Products
+----------+------------+--------+-------------+
|product_id|product_name|aisle_id|department_id|
+----------+------------+--------+-------------+
|         0|           0|       0|            0|
+----------+------------+--------+-------------+


Aisles
+--------+-----+
|aisle_id|aisle|
+--------+-----+
|       0|    0|
+--------+-----+


Departments
+-------------+----------+
|department_id|department|
+-------------+----------+
|            0|         0|
+-------------+----------+


Order Train
+--------+---------

In [0]:
# Duplicate Records

print("Orders Duplicates:", orders.count() - orders.dropDuplicates().count())
print("Products Duplicates:", products.count() - products.dropDuplicates().count())
print("Order Prior Duplicates:", order_prior.count() - order_prior.dropDuplicates().count())

Orders Duplicates: 0
Products Duplicates: 0
Order Prior Duplicates: 0


### Data Cleaning

In [0]:
# Remove Rows with Missing Values
orders = orders.dropna()
products = products.dropna()
aisles = aisles.dropna()
departments = departments.dropna()
order_train = order_train.dropna()
order_prior = order_prior.dropna()

In [0]:
print("Orders:", orders.count())
print("Products:", products.count())
print("Aisles:", aisles.count())
print("Departments:", departments.count())
print("Order Train:", order_train.count())
print("Order Prior:", order_prior.count())

Orders: 3214874
Products: 49688
Aisles: 134
Departments: 21
Order Train: 1384617
Order Prior: 1048575


## Table Joining

In [0]:
# Join Orders with Prior Orders
final_df = orders.join(order_prior, on="order_id", how="inner")

# Join Products
final_df = final_df.join(products, on="product_id", how="inner")

# Join Aisles
final_df = final_df.join(aisles, on="aisle_id", how="inner")

# Join Departments
final_df = final_df.join(departments, on="department_id", how="inner")

In [0]:
final_df.show(5)

+-------------+--------+----------+--------+-------+--------+------------+---------+-----------------+----------------------+-----------------+---------+--------------------+-------------+----------+
|department_id|aisle_id|product_id|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|add_to_cart_order|reordered|        product_name|        aisle|department|
+-------------+--------+----------+--------+-------+--------+------------+---------+-----------------+----------------------+-----------------+---------+--------------------+-------------+----------+
|            1|      37|      2043|   28252|  14121|   prior|          13|        3|               18|                   2.0|                7|        0|Organic Cookies '...|ice cream ice|    frozen|
|            7|      31|     38200|   63709|   3645|   prior|           2|        0|               17|                  26.0|                5|        0|         Apple Juice| refrigerated| beverages|


In [0]:
final_df.printSchema()

root
 |-- department_id: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle: string (nullable = true)
 |-- department: string (nullable = true)



In [0]:
print("Total Records:", final_df.count())

Total Records: 981545


#  Delta Table

In [0]:
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("grocery_analysis")

In [0]:
display(spark.table("grocery_analysis"))

department_id,aisle_id,product_id,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,product_name,aisle,department
1,37,2043,28252,14121,prior,13,3,18,2.0,7,0,Organic Cookies 'n Cream Ice Cream,ice cream ice,frozen
7,31,38200,63709,3645,prior,2,0,17,26.0,5,0,Apple Juice,refrigerated,beverages
16,120,10374,60318,154,prior,29,0,10,4.0,24,1,Flip Peanut Butter Dream Greek Yogurt,yogurt,dairy eggs
16,36,42736,26020,20261,prior,67,5,18,5.0,6,0,Unsalted Butter,butter,dairy eggs
1,37,15709,55141,13263,prior,11,5,15,0.0,2,0,Chocolate Grand Ice Cream,ice cream ice,frozen
4,83,21938,62939,10221,prior,24,3,15,3.0,9,1,Green Bell Pepper,fresh vegetables,produce
16,120,28156,20529,25600,prior,13,3,8,7.0,5,0,Total 0% Nonfat Plain Greek Yogurt,yogurt,dairy eggs
17,54,8021,73345,18165,prior,15,1,14,10.0,12,1,100% Recycled Paper Towels,paper goods,household
4,24,47626,15828,21284,prior,34,6,16,4.0,8,1,Large Lemon,fresh fruits,produce
16,21,22882,108231,35314,prior,32,1,23,8.0,16,1,Light Mozzarella String Cheese,packaged cheese,dairy eggs


In [0]:
spark.sql("SELECT * FROM grocery_analysis LIMIT 10").show()

+-------------+--------+----------+--------+-------+--------+------------+---------+-----------------+----------------------+-----------------+---------+--------------------+----------------+----------+
|department_id|aisle_id|product_id|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|add_to_cart_order|reordered|        product_name|           aisle|department|
+-------------+--------+----------+--------+-------+--------+------------+---------+-----------------+----------------------+-----------------+---------+--------------------+----------------+----------+
|            1|      37|      2043|   28252|  14121|   prior|          13|        3|               18|                   2.0|                7|        0|Organic Cookies '...|   ice cream ice|    frozen|
|            7|      31|     38200|   63709|   3645|   prior|           2|        0|               17|                  26.0|                5|        0|         Apple Juice|    refrigerat

## Business Analysis

In [0]:
# Top 10 Selling Products

from pyspark.sql.functions import count

top_products = final_df.groupBy("product_name") \
    .agg(count("product_id").alias("Total_Orders")) \
    .orderBy("Total_Orders", ascending=False)

display(top_products.limit(10))

product_name,Total_Orders
Banana,14460
Bag of Organic Bananas,11795
Organic Strawberries,8001
Organic Baby Spinach,7386
Organic Hass Avocado,6498
Organic Avocado,5183
Large Lemon,4679
Strawberries,4301
Limes,4254
Organic Raspberries,4200


In [0]:
# Top 10 Departments

top_departments = final_df.groupBy("department") \
    .agg(count("order_id").alias("Total_Orders")) \
    .orderBy("Total_Orders", ascending=False)

display(top_departments)

department,Total_Orders
produce,287696
dairy eggs,164360
snacks,87761
beverages,81578
frozen,67160
pantry,56012
bakery,35643
canned goods,32103
deli,31813
dry goods pasta,25717


In [0]:
# Top 10 Aisles

top_aisles = final_df.groupBy("aisle") \
    .agg(count("order_id").alias("Total_Orders")) \
    .orderBy("Total_Orders", ascending=False)

display(top_aisles.limit(10))

aisle,Total_Orders
fresh fruits,110862
fresh vegetables,103379
packaged vegetables fruits,53806
yogurt,43949
packaged cheese,29596
milk,27155
water seltzer sparkling water,25401
chips pretzels,21764
soy lactosefree,19497
bread,17749


In [0]:
# Most Reordered Products

from pyspark.sql.functions import sum

reordered_products = final_df.groupBy("product_name") \
    .agg(sum("reordered").alias("Reorder_Count")) \
    .orderBy("Reorder_Count", ascending=False)

display(reordered_products.limit(10))

product_name,Reorder_Count
Banana,13024
Bag of Organic Bananas,10297
Organic Strawberries,6603
Organic Baby Spinach,6093
Organic Hass Avocado,5435
Organic Avocado,4349
Organic Whole Milk,3615
Large Lemon,3456
Organic Raspberries,3361
Strawberries,3204


In [0]:
# Orders by Hour

orders_by_hour = final_df.groupBy("order_hour_of_day") \
    .agg(count("order_id").alias("Total_Orders")) \
    .orderBy("order_hour_of_day")

display(orders_by_hour)

order_hour_of_day,Total_Orders
0,6529
1,3718
2,2125
3,1672
4,1423
5,2739
6,9227
7,27213
8,54063
9,75298


In [0]:
# Average Basket Size

from pyspark.sql.functions import avg

basket_size = final_df.groupBy("order_id") \
    .agg(count("product_id").alias("Products_Per_Order"))
basket_size.select(avg("Products_Per_Order").alias("Average_Basket_Size")).show()

+-------------------+
|Average_Basket_Size|
+-------------------+
|  10.08036191102165|
+-------------------+



## SQL Analysis

In [0]:
%sql
-- Top 10 Customers

SELECT
user_id,
COUNT(DISTINCT order_id) AS Total_Orders
FROM grocery_analysis
GROUP BY user_id
ORDER BY Total_Orders DESC
LIMIT 10;

user_id,Total_Orders
34340,12
140395,11
22104,10
169586,10
201248,10
178209,10
74642,9
60932,9
108495,9
171807,9


In [0]:
%sql
-- Orders by Day of Week

SELECT
order_dow,
COUNT(order_id) AS Total_Orders
FROM grocery_analysis
GROUP BY order_dow
ORDER BY order_dow;

order_dow,Total_Orders
0,186261
1,170717
2,129198
3,116459
4,114584
5,127532
6,136794


In [0]:
%sql
-- First Product Added to Cart

SELECT
product_name,
COUNT(*) AS First_Add_Count
FROM grocery_analysis
WHERE add_to_cart_order = 1
GROUP BY product_name
ORDER BY First_Add_Count DESC
LIMIT 10;

product_name,First_Add_Count
Banana,3350
Bag of Organic Bananas,2449
Organic Whole Milk,928
Organic Strawberries,811
Organic Baby Spinach,741
Organic Hass Avocado,711
Organic Avocado,653
Strawberries,492
Spring Water,476
Organic Raspberries,438


In [0]:
%sql
-- Final KPI Summary

SELECT
COUNT(DISTINCT order_id) AS Total_Orders,
COUNT(DISTINCT user_id) AS Total_Customers,
COUNT(DISTINCT product_id) AS Total_Products,
ROUND(AVG(days_since_prior_order),2) AS Avg_Days_Between_Orders
FROM grocery_analysis;

Total_Orders,Total_Customers,Total_Products,Avg_Days_Between_Orders
97372,64636,34763,11.03
